In [ ]:
"""
HPS Tau Trigger Fake Rate Analysis
- Fraction of HLT-passing reco taus that fail gen-matching (non-matched = fake taus)
- Events separated by decay mode (DM) groups
- Events separated by number of additional GenJets not overlapping with GenVisTau

gen_match logic (inverted for fake rate, opposite of efficiency)
    events_flag = True  → gen-match failed  = FAKE tau
    events_flag = False → gen-match succeeded = REAL tau

DM_GROUPS  : groups by decay mode
JET_GROUPS : groups by number of additional GenJets not overlapping with GenVisTau
"""

import os
import math
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
import uproot
import vector
import mplhep as hep

vector.register_awkward()


# ══════════════════════════════════════════════════════════════════════════════
# 0. Bin / Group Definitions
# ══════════════════════════════════════════════════════════════════════════════
_b0 = np.arange(0,   130, 10)
_b1 = np.arange(131, 180,  5)
_b2 = np.arange(181, 300, 20)
_b3 = np.arange(301, 601, 100)
BINS_PT  = np.concatenate((_b0, _b1, _b2, _b3))
BINS_ETA = np.arange(-3.0, 3.0, 0.1)
BINS_PHI = np.arange(-3.2, 3.2, 0.1)
PT_CUT   = 150

# Decay mode groups
DM_GROUPS = {
    "all":         {"modes": [0,1,2,10,11,15], "label": r"$\tau$ (all DM)"},
    "1Prong_DM0":  {"modes": [0],               "label": r"1-Prong DM0"},
    "1Prong_DM1":  {"modes": [1],               "label": r"1-Prong DM1"},
    "1Prong_DM2":  {"modes": [2],               "label": r"1-Prong DM2"},
    "3Prong_DM10": {"modes": [10],              "label": r"3-Prong DM10"},
    "3Prong_DM11": {"modes": [11],              "label": r"3-Prong DM11"},
    "5Prong_DM15": {"modes": [15],              "label": r"5-Prong DM15"},
    "1Prong_all":  {"modes": [0,1,2],           "label": r"1-Prong (all)"},
    "3Prong_all":  {"modes": [10,11],           "label": r"3-Prong (all)"},
    "5Prong_all":  {"modes": [15],              "label": r"5-Prong (all)"},
}

# Groups by number of additional GenJets not overlapping with GenVisTau
JET_GROUPS = {
    "all":    {
        "selector": lambda n: n >= 0,
        "label":    "All",
        "color":    "#7B2D8B",
    },
    "0jet":   {
        "selector": lambda n: n == 0,
        "label":    r"$n_{\rm jet}^{\rm add}=0$",
        "color":    "#4C78A8",
    },
    "1jet":   {
        "selector": lambda n: n == 1,
        "label":    r"$n_{\rm jet}^{\rm add}=1$",
        "color":    "#F58518",
    },
    "ge2jet": {
        "selector": lambda n: n >= 2,
        "label":    r"$n_{\rm jet}^{\rm add}\geq 2$",
        "color":    "#54A24B",
    },
}


# ══════════════════════════════════════════════════════════════════════════════
# 1. Utilities
# ══════════════════════════════════════════════════════════════════════════════
def deltaR(eta1, phi1, eta2, phi2):
    deta = eta1 - eta2
    dphi = (phi1 - phi2 + math.pi) % (2 * math.pi) - math.pi
    return math.sqrt(deta**2 + dphi**2)


def make_dm_mask(dm_arr, modes):
    mask = np.zeros(len(dm_arr), dtype=bool)
    for m in modes:
        mask |= (dm_arr == m)
    return mask


def extract_leading_genvis_dm(genvis_pt, genvis_dm, mask):
    """
    Returns the leading GenVisTau decay mode (GenVisTau_status) for events selected by mask.
    Returns -1 for events with no GenVisTau.

    Parameters
    ----------
    genvis_pt : ak.Array  shape (n_events, var)
    genvis_dm : ak.Array  shape (n_events, var)
    mask      : np.ndarray[bool] length n_events

    Returns
    -------
    np.ndarray[int]  length = mask.sum()
    """
    result = []
    for i in np.where(mask)[0]:
        gv_pt_i = genvis_pt[i]
        if len(gv_pt_i) == 0:
            result.append(-1)
        else:
            best_idx = int(ak.argmax(gv_pt_i))
            result.append(int(genvis_dm[i][best_idx]))
    return np.array(result, dtype=int)


def ratio_and_err(num_counts, den_counts):
    ratio = np.divide(num_counts, den_counts,
                      out=np.zeros_like(num_counts, dtype=float),
                      where=(den_counts != 0))
    err = np.where(den_counts > 0,
                   np.sqrt(ratio * (1 - ratio) / np.maximum(den_counts, 1)), 0)
    return ratio, err


def _save_or_show(fig, save_dir, filename):
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, filename), dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"  saved → {filename}")
    else:
        plt.show()
        plt.close(fig)


# ══════════════════════════════════════════════════════════════════════════════
# 2. Data Loading
# ══════════════════════════════════════════════════════════════════════════════
def get_info(sample):
    file   = uproot.open(sample)
    events = file["Events"]

    tau_pt   = events["hltHpsPFTau_pt"].array()
    tau_eta  = events["hltHpsPFTau_eta"].array()
    tau_phi  = events["hltHpsPFTau_phi"].array()
    tau_mass = events["hltHpsPFTau_mass"].array()

    # Decay mode: based on GenVisTau_status (not reco tau DM)
    genvis_tau_dm = events["GenVisTau_status"].array()

    genvis_tau_pt   = events["GenVisTau_pt"].array()
    genvis_tau_eta  = events["GenVisTau_eta"].array()
    genvis_tau_phi  = events["GenVisTau_phi"].array()
    genvis_tau_mass = events["GenVisTau_mass"].array()

    genjet_pt  = events["GenJet_pt"].array()
    genjet_eta = events["GenJet_eta"].array()
    genjet_phi = events["GenJet_phi"].array()

    tau_trigger = events["HLT_LooseDeepTauPFTauHPS180_L2NN_eta2p1"].array()

    # Event filter: reco tau >= 1 AND GenVisTau == 1
    has_tau        = ak.num(tau_pt) > 0
    has_one_genvis = ak.num(genvis_tau_pt) == 1
    evt_mask       = np.asarray(has_tau & has_one_genvis)

    tau_pt   = tau_pt[evt_mask];   tau_eta  = tau_eta[evt_mask]
    tau_phi  = tau_phi[evt_mask];  tau_mass = tau_mass[evt_mask]
    genvis_tau_dm = genvis_tau_dm[evt_mask]

    genvis_tau_pt   = genvis_tau_pt[evt_mask]
    genvis_tau_eta  = genvis_tau_eta[evt_mask]
    genvis_tau_phi  = genvis_tau_phi[evt_mask]
    genvis_tau_mass = genvis_tau_mass[evt_mask]

    genjet_pt  = genjet_pt[evt_mask]
    genjet_eta = genjet_eta[evt_mask]
    genjet_phi = genjet_phi[evt_mask]

    tau_trigger = tau_trigger[evt_mask]

    print(f"total events: {int(ak.sum(has_tau))} → after filter: {len(tau_trigger)}")

    return (events,
            tau_pt, tau_eta, tau_phi, tau_mass,
            genvis_tau_pt, genvis_tau_eta, genvis_tau_phi, genvis_tau_mass, genvis_tau_dm,
            tau_trigger,
            genjet_pt, genjet_eta, genjet_phi)


# ══════════════════════════════════════════════════════════════════════════════
# 3. Gen Matching  (inverted logic for fake rate)
# ══════════════════════════════════════════════════════════════════════════════
def gen_match(tau_pt, tau_eta, tau_phi,
              genvis_tau_pt, genvis_tau_eta, genvis_tau_phi,
              deltaR_threshold=0.1):
    """
    ※ Inverted logic for fake rate ※
        events_flag = True  → gen-match failed  = FAKE tau
        events_flag = False → gen-match succeeded = REAL tau
    """
    events_flag     = []
    matched_gen_pt  = []
    matched_gen_eta = []
    matched_gen_phi = []

    for i in range(len(tau_pt)):
        if len(tau_pt[i]) == 0:
            events_flag.append(False)
            matched_gen_pt.append(np.nan)
            matched_gen_eta.append(np.nan)
            matched_gen_phi.append(np.nan)
            continue

        idx      = ak.argsort(tau_pt[i], ascending=False)
        lead_eta = tau_eta[i][idx[0]]
        lead_phi = tau_phi[i][idx[0]]

        if abs(lead_eta) > 2.1:
            events_flag.append(False)
            matched_gen_pt.append(np.nan)
            matched_gen_eta.append(np.nan)
            matched_gen_phi.append(np.nan)
            continue

        best_dR  = 999.
        best_idx = -1
        for j in range(len(genvis_tau_pt[i])):
            dR = deltaR(float(lead_eta), float(lead_phi),
                        float(genvis_tau_eta[i][j]),
                        float(genvis_tau_phi[i][j]))
            if dR < deltaR_threshold and dR < best_dR:
                best_dR  = dR
                best_idx = j

        if best_idx >= 0:
            events_flag.append(False)    # REAL
            matched_gen_pt.append(float(genvis_tau_pt[i][best_idx]))
            matched_gen_eta.append(float(genvis_tau_eta[i][best_idx]))
            matched_gen_phi.append(float(genvis_tau_phi[i][best_idx]))
        else:
            events_flag.append(True)     # FAKE
            matched_gen_pt.append(np.nan)
            matched_gen_eta.append(np.nan)
            matched_gen_phi.append(np.nan)

    return events_flag, matched_gen_pt, matched_gen_eta, matched_gen_phi


# ══════════════════════════════════════════════════════════════════════════════
# 4. Additional GenJet Count (excluding GenVisTau overlap)
# ══════════════════════════════════════════════════════════════════════════════
def count_additional_genjets(genvis_tau_eta, genvis_tau_phi,
                             genjet_pt, genjet_eta, genjet_phi,
                             deltaR_threshold=0.3,
                             genjet_pt_cut=30.0, genjet_eta_cut=2.5):
    """
    Returns the number of additional GenJets in each event that do not overlap
    (ΔR >= threshold) with any GenVisTau.

    Returns
    -------
    np.ndarray[int]  length = number of events
    """
    n_arr = []
    for i in range(len(genjet_eta)):
        g_eta_i = np.array(genvis_tau_eta[i])
        g_phi_i = np.array(genvis_tau_phi[i])

        jet_pt_i  = np.array(genjet_pt[i])
        jet_eta_i = np.array(genjet_eta[i])
        jet_phi_i = np.array(genjet_phi[i])

        # basic kinematic cuts
        sel = (jet_pt_i >= genjet_pt_cut) & (np.abs(jet_eta_i) <= genjet_eta_cut)
        jet_eta_i = jet_eta_i[sel]
        jet_phi_i = jet_phi_i[sel]

        count = 0
        for j in range(len(jet_eta_i)):
            j_eta = float(jet_eta_i[j])
            j_phi = float(jet_phi_i[j])
            # must not overlap with any GenVisTau
            is_isolated = all(
                deltaR(float(g_eta_i[t]), float(g_phi_i[t]), j_eta, j_phi) >= deltaR_threshold
                for t in range(len(g_eta_i))
            )
            if is_isolated:
                count += 1

        n_arr.append(count)

    return np.array(n_arr, dtype=int)


# ══════════════════════════════════════════════════════════════════════════════
# 5. Numerator / Denominator Calculation
# ══════════════════════════════════════════════════════════════════════════════
def returns_num_den(events_flag, matched_gen_pt, matched_gen_eta, matched_gen_phi,
                    final_tau_pt, final_tau_eta, final_tau_phi,
                    tau_trigger_filter,
                    genvis_tau_pt, genvis_tau_dm,
                    nearby_jet_counts=None):
    """
    Den = all events (all events with at least one reco tau)
    Num = fake (gen-match failed) AND HLT trigger passed
    """
    events_flag_arr = np.array(events_flag, dtype=bool)
    tau_trigger_arr = np.array(tau_trigger_filter, dtype=bool)

    njets_arr = (np.array(nearby_jet_counts, dtype=int)
                 if nearby_jet_counts is not None
                 else np.zeros(len(events_flag_arr), dtype=int))

    den_mask = np.ones(len(events_flag_arr), dtype=bool)
    num_mask = events_flag_arr & tau_trigger_arr

    def extract_leading(pt_src, kin_src, mask):
        pt_sel  = pt_src[mask]
        kin_sel = kin_src[mask]
        idx     = ak.argsort(pt_sel, axis=1, ascending=False)
        return ak.to_numpy(ak.flatten(kin_sel[idx[:, :1]], axis=1))

    den_pt_reco  = extract_leading(final_tau_pt, final_tau_pt,  den_mask)
    den_eta_reco = extract_leading(final_tau_pt, final_tau_eta, den_mask)
    den_phi_reco = extract_leading(final_tau_pt, final_tau_phi, den_mask)
    num_pt_reco  = extract_leading(final_tau_pt, final_tau_pt,  num_mask)
    num_eta_reco = extract_leading(final_tau_pt, final_tau_eta, num_mask)
    num_phi_reco = extract_leading(final_tau_pt, final_tau_phi, num_mask)

    # DM: based on GenVisTau_status, decay mode of the leading GenVisTau (highest pT)
    den_dm = extract_leading_genvis_dm(genvis_tau_pt, genvis_tau_dm, den_mask)
    num_dm = extract_leading_genvis_dm(genvis_tau_pt, genvis_tau_dm, num_mask)

    den_njets = njets_arr[den_mask]
    num_njets = njets_arr[num_mask]

    print(f"\nDen (all reco tau events): {den_mask.sum()}")
    print(f"Num (fake + triggered):    {num_mask.sum()}")
    for tag, arr in [("Den", den_njets), ("Num", num_njets)]:
        print(f"  {tag}: n_jet=0: {(arr==0).sum():4d} | "
              f"n_jet=1: {(arr==1).sum():4d} | "
              f"n_jet≥2: {(arr>=2).sum():4d}")

    return (num_pt_reco,  den_pt_reco,
            num_eta_reco, den_eta_reco,
            num_phi_reco, den_phi_reco,
            num_dm,       den_dm,
            num_njets,    den_njets)


# ══════════════════════════════════════════════════════════════════════════════
# 6. Fake Rate Plots (DM × Jet count groups)
# ══════════════════════════════════════════════════════════════════════════════
def _draw_fakerate_ax(ax, bc, hw, ratio, err, color, label):
    ax.errorbar(bc, ratio, xerr=hw, yerr=err,
                fmt='o', color=color, ecolor=color,
                capsize=3, alpha=0.85, label=label)


def run_all_fakerate(
    num_pt_reco,  den_pt_reco,
    num_eta_reco, den_eta_reco,
    num_phi_reco, den_phi_reco,
    num_dm_arr,   den_dm_arr,
    num_njets,    den_njets,
    pt_cut=PT_CUT,
    save_dir=None,
    dm_keys=None,
    jet_keys=None,
    overlay=True,
):
    """
    Generates pT / eta / phi fake rate plots for each DM group × Jet count group combination.

    File naming when overlay=True:
        {dm_key}_overlay_fakerate_{pt|eta|phi}.png
        e.g.: all__overlay_fakerate_pt.png

    File naming when overlay=False:
        {dm_key}__{jet_key}_fakerate_{pt|eta|phi}.png
        e.g.: 3Prong_all__1jet_fakerate_pt.png
    """
    plt.style.use(hep.style.CMS)
    HEADER = r"$Z'(500\,\mathrm{GeV})\to\tau\tau$, PU=200"
    results = {}

    _dm_iter  = dm_keys  if dm_keys  is not None else list(DM_GROUPS.keys())
    _jet_iter = jet_keys if jet_keys is not None else list(JET_GROUPS.keys())

    for dm_key in _dm_iter:
        if dm_key not in DM_GROUPS:
            continue
        dm_info  = DM_GROUPS[dm_key]
        dm_label = dm_info["label"]
        modes    = dm_info["modes"]

        dm_num_mask = make_dm_mask(num_dm_arr, modes)
        dm_den_mask = make_dm_mask(den_dm_arr, modes)

        _num_pt_dm  = num_pt_reco[dm_num_mask]
        _den_pt_dm  = den_pt_reco[dm_den_mask]
        _num_eta_dm = num_eta_reco[dm_num_mask]
        _den_eta_dm = den_eta_reco[dm_den_mask]
        _num_phi_dm = num_phi_reco[dm_num_mask]
        _den_phi_dm = den_phi_reco[dm_den_mask]
        _num_nj_dm  = num_njets[dm_num_mask]
        _den_nj_dm  = den_njets[dm_den_mask]

        if dm_den_mask.sum() == 0:
            print(f"  [{dm_key}] no events, skipping")
            continue

        info_str = HEADER + "\n" + dm_label

        # ── prepare overlay plots ──────────────────────────────────────────
        if overlay:
            fig_pt,  ax_pt  = plt.subplots(figsize=(10, 7))
            fig_eta, ax_eta = plt.subplots(figsize=(10, 7))
            fig_phi, ax_phi = plt.subplots(figsize=(10, 7))
            for ax, xlabel, xlim in [
                (ax_pt,  r'Tau $p_T$ [GeV]',
                 (BINS_PT[0], BINS_PT[-1])),
                (ax_eta, rf'Tau $\eta$  ($p_T>{pt_cut}$ GeV)',
                 (BINS_ETA[0], BINS_ETA[-1])),
                (ax_phi, rf'Tau $\phi$  ($p_T>{pt_cut}$ GeV)',
                 (BINS_PHI[0], BINS_PHI[-1])),
            ]:
                ax.set_ylabel('Fake rate', fontsize=16)
                ax.set_xlabel(xlabel, fontsize=16)
                ax.set_xlim(*xlim)
                ax.set_ylim(0, 1.15)
                ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
                ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
                hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
                ax.text(0.97, 1.02, info_str, transform=ax.transAxes,
                        fontsize=11, va='bottom', ha='right',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
            ax_pt.axvline(x=pt_cut, color='gray', linestyle='--',
                          linewidth=2, label=f'Threshold ({pt_cut} GeV)')

        # ── Jet group loop ─────────────────────────────────────────────────
        for jet_key in _jet_iter:
            if jet_key not in JET_GROUPS:
                continue
            jet_info  = JET_GROUPS[jet_key]
            jet_label = jet_info["label"]
            color     = jet_info["color"]
            jet_sel   = jet_info["selector"]

            j_num = jet_sel(_num_nj_dm)
            j_den = jet_sel(_den_nj_dm)

            if j_den.sum() == 0:
                print(f"  [{dm_key} / {jet_key}] no events, skipping")
                continue

            _num_pt  = _num_pt_dm[j_num];   _den_pt  = _den_pt_dm[j_den]
            _num_eta = _num_eta_dm[j_num];   _den_eta = _den_eta_dm[j_den]
            _num_phi = _num_phi_dm[j_num];   _den_phi = _den_phi_dm[j_den]

            tag      = f"{dm_key}__{jet_key}"
            full_str = info_str + f",  {jet_label}"

            # ── pT fake rate ──────────────────────────────────────────────
            c_n, edges = np.histogram(_num_pt, bins=BINS_PT)
            c_d, _     = np.histogram(_den_pt, bins=BINS_PT)
            r, e       = ratio_and_err(c_n, c_d)
            bc = (edges[:-1] + edges[1:]) / 2
            hw = (edges[1:]  - edges[:-1]) / 2

            if overlay:
                _draw_fakerate_ax(ax_pt, bc, hw, r, e, color, jet_label)
            else:
                fig, ax = plt.subplots(figsize=(10, 7))
                _draw_fakerate_ax(ax, bc, hw, r, e, color, jet_label)
                ax.set_xlabel(r'Tau $p_T$ [GeV]', fontsize=16)
                ax.set_ylabel('Fake rate', fontsize=16)
                ax.set_xlim(edges[0], edges[-1]);  ax.set_ylim(0, 1.15)
                ax.axvline(x=pt_cut, color='gray', linestyle='--', linewidth=2,
                           label=f'Threshold ({pt_cut} GeV)')
                ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
                ax.legend(fontsize=12, loc='upper right')
                ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
                hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
                ax.text(0.97, 1.02, full_str, transform=ax.transAxes,
                        fontsize=11, va='bottom', ha='right',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
                plt.tight_layout()
                _save_or_show(fig, save_dir, f"{tag}_fakerate_pt.png")

            # ── eta fake rate (pT > pt_cut) ───────────────────────────────
            num_eta_cut = _num_eta[_num_pt >= pt_cut]
            den_eta_cut = _den_eta[_den_pt >= pt_cut]
            c_n, edges  = np.histogram(num_eta_cut, bins=BINS_ETA)
            c_d, _      = np.histogram(den_eta_cut, bins=BINS_ETA)
            r_eta, e_eta = ratio_and_err(c_n, c_d)
            bc_e = (edges[:-1] + edges[1:]) / 2
            hw_e = (edges[1:]  - edges[:-1]) / 2

            if overlay:
                _draw_fakerate_ax(ax_eta, bc_e, hw_e, r_eta, e_eta, color, jet_label)
            else:
                fig, ax = plt.subplots(figsize=(10, 7))
                _draw_fakerate_ax(ax, bc_e, hw_e, r_eta, e_eta, color, jet_label)
                ax.set_xlabel(rf'Tau $\eta$  ($p_T>{pt_cut}$ GeV)', fontsize=16)
                ax.set_ylabel('Fake rate', fontsize=16)
                ax.set_xlim(BINS_ETA[0], BINS_ETA[-1]);  ax.set_ylim(0, 1.15)
                ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
                ax.legend(fontsize=12, loc='upper right')
                ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
                hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
                ax.text(0.97, 1.02, full_str, transform=ax.transAxes,
                        fontsize=11, va='bottom', ha='right',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
                plt.tight_layout()
                _save_or_show(fig, save_dir, f"{tag}_fakerate_eta.png")

            # ── phi fake rate (pT > pt_cut) ───────────────────────────────
            num_phi_cut = _num_phi[_num_pt >= pt_cut]
            den_phi_cut = _den_phi[_den_pt >= pt_cut]
            c_n, edges  = np.histogram(num_phi_cut, bins=BINS_PHI)
            c_d, _      = np.histogram(den_phi_cut, bins=BINS_PHI)
            r_phi, e_phi = ratio_and_err(c_n, c_d)
            bc_p = (edges[:-1] + edges[1:]) / 2
            hw_p = (edges[1:]  - edges[:-1]) / 2

            if overlay:
                _draw_fakerate_ax(ax_phi, bc_p, hw_p, r_phi, e_phi, color, jet_label)
            else:
                fig, ax = plt.subplots(figsize=(10, 7))
                _draw_fakerate_ax(ax, bc_p, hw_p, r_phi, e_phi, color, jet_label)
                ax.set_xlabel(rf'Tau $\phi$  ($p_T>{pt_cut}$ GeV)', fontsize=16)
                ax.set_ylabel('Fake rate', fontsize=16)
                ax.set_xlim(BINS_PHI[0], BINS_PHI[-1]);  ax.set_ylim(0, 1.15)
                ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
                ax.legend(fontsize=12, loc='upper right')
                ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
                hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
                ax.text(0.97, 1.02, full_str, transform=ax.transAxes,
                        fontsize=11, va='bottom', ha='right',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
                plt.tight_layout()
                _save_or_show(fig, save_dir, f"{tag}_fakerate_phi.png")

            # ── Plateau fake rate ─────────────────────────────────────────
            n_n = (_num_pt >= pt_cut).sum()
            n_d = (_den_pt >= pt_cut).sum()
            fr  = n_n / n_d * 100 if n_d > 0 else 0.
            err = np.sqrt(fr / 100 * (1 - fr / 100) / max(n_d, 1)) * 100
            results[tag] = dict(dm_label=dm_label, jet_label=jet_label,
                                n_num=n_n, n_den=n_d,
                                fake_rate=fr, err=err)

        # ── finalize overlay (save 3 plots for this DM group) ─────────────
        if overlay:
            for ax in (ax_pt, ax_eta, ax_phi):
                ax.legend(loc='upper right', fontsize=11, frameon=True)
            plt.figure(fig_pt.number);  plt.tight_layout()
            plt.figure(fig_eta.number); plt.tight_layout()
            plt.figure(fig_phi.number); plt.tight_layout()
            _save_or_show(fig_pt,  save_dir, f"{dm_key}__overlay_fakerate_pt.png")
            _save_or_show(fig_eta, save_dir, f"{dm_key}__overlay_fakerate_eta.png")
            _save_or_show(fig_phi, save_dir, f"{dm_key}__overlay_fakerate_phi.png")

    # ── summary output ───────────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print(f"  Plateau Fake Rate  (pT > {pt_cut} GeV)")
    print(f"{'='*80}")
    print(f"{'DM':<18} {'Jet group':<36} {'num/den':<14} {'Fake Rate'}")
    print(f"{'-'*80}")
    for tag, r in results.items():
        print(f"{r['dm_label']:<18} {r['jet_label']:<36} "
              f"{r['n_num']:>5}/{r['n_den']:<8} "
              f"{r['fake_rate']:>6.2f} ± {r['err']:>5.2f} %")
    print(f"{'='*80}\n")

    return results


# ══════════════════════════════════════════════════════════════════════════════
# 7. Main Execution
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":

    SAMPLE   = "/gv0/Users/achihwan/phase2/cmssw_16/condor/hltrun/new_150.root"
    SAVE_DIR = "./260521/1GenvisTau/"
    os.makedirs(SAVE_DIR, exist_ok=True)

    # ── 1) Data loading ──────────────────────────────────────────────────────
    (events_150,
     tau_pt_150, tau_eta_150, tau_phi_150, tau_mass_150,
     genvis_tau_pt_150, genvis_tau_eta_150, genvis_tau_phi_150, genvis_tau_mass_150, genvis_tau_dm_150,
     tau_trigger_150,
     genjet_pt_150, genjet_eta_150, genjet_phi_150) = get_info(SAMPLE)

    # ── 2) Gen matching (inverted logic for fake rate) ────────────────────────
    (events_flag_150,
     matched_gen_pt_150,
     matched_gen_eta_150,
     matched_gen_phi_150) = gen_match(
        tau_pt_150, tau_eta_150, tau_phi_150,
        genvis_tau_pt_150, genvis_tau_eta_150, genvis_tau_phi_150,
    )

    # ── 3) Count additional GenJets not overlapping with GenVisTau ────────────
    additional_jet_counts_150 = count_additional_genjets(
        genvis_tau_eta_150, genvis_tau_phi_150,
        genjet_pt_150, genjet_eta_150, genjet_phi_150,
        deltaR_threshold=0.3,
        genjet_pt_cut=30.0,
        genjet_eta_cut=2.5,
    )

    n = additional_jet_counts_150
    print(f"\n[Additional gen jet count (excluding taus, pT>30, |eta|<2.5)]")
    print(f"  n_jet = 0  : {(n == 0).sum()}")
    print(f"  n_jet = 1  : {(n == 1).sum()}")
    print(f"  n_jet >= 2 : {(n >= 2).sum()}")

    # ── 4) Numerator / Denominator calculation ────────────────────────────────
    (num_pt_reco_150,  den_pt_reco_150,
     num_eta_reco_150, den_eta_reco_150,
     num_phi_reco_150, den_phi_reco_150,
     num_dm_150,       den_dm_150,
     num_njets_150,    den_njets_150) = returns_num_den(
        events_flag_150,
        matched_gen_pt_150, matched_gen_eta_150, matched_gen_phi_150,
        tau_pt_150, tau_eta_150, tau_phi_150,
        tau_trigger_150,
        genvis_tau_pt_150, genvis_tau_dm_150,
        nearby_jet_counts=additional_jet_counts_150,
    )

    # ── 5) Generate fake rate plots ───────────────────────────────────────────
    # overlay=True  → for each DM group, overlay 4 jet groups on one plot for comparison
    # overlay=False → save individual files for each DM × jet group combination
    results = run_all_fakerate(
        num_pt_reco_150,  den_pt_reco_150,
        num_eta_reco_150, den_eta_reco_150,
        num_phi_reco_150, den_phi_reco_150,
        num_dm_150,       den_dm_150,
        num_njets_150,    den_njets_150,
        pt_cut=150,
        save_dir=SAVE_DIR,
        overlay=True,
        # To plot only specific groups, uncomment:
        # dm_keys=["all", "1Prong_all", "2Prong_all", "3Prong_all"],
        # jet_keys=["all", "0jet", "1jet", "ge2jet"],
    )

In [ ]:
def run_all_numden(
    num_pt_reco,  den_pt_reco,
    num_eta_reco, den_eta_reco,
    num_phi_reco, den_phi_reco,
    num_dm_arr,   den_dm_arr,
    num_njets,    den_njets,
    pt_cut=PT_CUT,
    save_dir=None,
    dm_keys=None,
    jet_keys=None,
):
    plt.style.use(hep.style.CMS)
    HEADER = r"$Z'(500\,\mathrm{GeV})\to\tau\tau$, PU=200"

    _dm_iter  = dm_keys  if dm_keys  is not None else list(DM_GROUPS.keys())
    _jet_iter = jet_keys if jet_keys is not None else list(JET_GROUPS.keys())

    DEN_COLOR = "#4C78A8"
    NUM_COLOR = "#E45756"

    def _hist_step_normed(ax, data, bins, color, label, linestyle='-'):
        counts, edges = np.histogram(data, bins=bins)
        widths        = edges[1:] - edges[:-1]
        heights       = counts / widths
        ax.step(edges[:-1], heights, where='post',
                color=color, linestyle=linestyle,
                linewidth=2.0, alpha=0.9, label=label)
        err = np.sqrt(counts) / widths
        bc  = (edges[:-1] + edges[1:]) / 2
        ax.errorbar(bc, heights, yerr=err,
                    fmt='none', color=color, capsize=2, alpha=0.7)

    for dm_key in _dm_iter:
        if dm_key not in DM_GROUPS:
            continue
        dm_info  = DM_GROUPS[dm_key]
        dm_label = dm_info["label"]
        modes    = dm_info["modes"]

        dm_num_mask = make_dm_mask(num_dm_arr, modes)
        dm_den_mask = make_dm_mask(den_dm_arr, modes)

        if dm_den_mask.sum() == 0:
            print(f"  [{dm_key}] no events, skipping")
            continue

        _num_pt_dm  = num_pt_reco[dm_num_mask]
        _den_pt_dm  = den_pt_reco[dm_den_mask]
        _num_eta_dm = num_eta_reco[dm_num_mask]
        _den_eta_dm = den_eta_reco[dm_den_mask]
        _num_phi_dm = num_phi_reco[dm_num_mask]
        _den_phi_dm = den_phi_reco[dm_den_mask]
        _num_nj_dm  = num_njets[dm_num_mask]
        _den_nj_dm  = den_njets[dm_den_mask]

        for jet_key in _jet_iter:
            if jet_key not in JET_GROUPS:
                continue
            jet_info  = JET_GROUPS[jet_key]
            jet_label = jet_info["label"]
            jet_sel   = jet_info["selector"]

            j_num = jet_sel(_num_nj_dm)
            j_den = jet_sel(_den_nj_dm)

            if j_den.sum() == 0:
                print(f"  [{dm_key} / {jet_key}] no events, skipping")
                continue

            _num_pt  = _num_pt_dm[j_num];   _den_pt  = _den_pt_dm[j_den]
            _num_eta = _num_eta_dm[j_num];   _den_eta = _den_eta_dm[j_den]
            _num_phi = _num_phi_dm[j_num];   _den_phi = _den_phi_dm[j_den]

            # ── |eta| < 2.1 mask ─────────────────────────────────
            eta_m_num = np.abs(_num_eta) < 2.1
            eta_m_den = np.abs(_den_eta) < 2.1

            tag      = f"{dm_key}__{jet_key}"
            info_str = HEADER + "\n" + dm_label + f",  {jet_label}"

            def _decorate(ax, xlabel, xlim):
                ax.set_ylabel(r'Events / bin width', fontsize=15)
                ax.set_xlabel(xlabel, fontsize=15)
                ax.set_xlim(*xlim)
                ax.set_ylim(bottom=0)
                ax.grid(which='both', linestyle=':', linewidth=0.5, alpha=0.5)
                hep.cms.text("Simulation Preliminary", loc=0, ax=ax)
                ax.text(0.97, 1.02, info_str,
                        transform=ax.transAxes, fontsize=11,
                        va='bottom', ha='right',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
                ax.legend(fontsize=12, loc='upper right')

            # ── pT  (|eta| < 2.1) ───────────────────────────────
            fig, ax = plt.subplots(figsize=(10, 7))
            _hist_step_normed(ax, _den_pt[eta_m_den], BINS_PT,
                              DEN_COLOR, 'Den (all reco tau)',  '--')
            _hist_step_normed(ax, _num_pt[eta_m_num], BINS_PT,
                              NUM_COLOR, 'Num (fake + HLT)',    '-')
            ax.axvline(x=pt_cut, color='gray', linestyle='--',
                       linewidth=1.5, label=f'pT cut ({pt_cut} GeV)')
            _decorate(ax, r'Tau $p_T$ [GeV]', (BINS_PT[0], BINS_PT[-1]))
            ax.set_ylabel(r'Events / GeV', fontsize=15)
            plt.tight_layout()
            _save_or_show(fig, save_dir, f"{tag}_numden_pt.png")

            # ── eta  (|eta| < 2.1, pT > pt_cut) ────────────────
            den_sel = eta_m_den & (_den_pt >= pt_cut)
            num_sel = eta_m_num & (_num_pt >= pt_cut)
            fig, ax = plt.subplots(figsize=(10, 7))
            _hist_step_normed(ax, _den_eta[den_sel], BINS_ETA,
                              DEN_COLOR, 'Den (all reco tau)',  '--')
            _hist_step_normed(ax, _num_eta[num_sel], BINS_ETA,
                              NUM_COLOR, 'Num (fake + HLT)',    '-')
            _decorate(ax, rf'Tau $\eta$  ($p_T>{pt_cut}$ GeV)',
                      (BINS_ETA[0], BINS_ETA[-1]))
            plt.tight_layout()
            _save_or_show(fig, save_dir, f"{tag}_numden_eta.png")

            # ── phi  (|eta| < 2.1, pT > pt_cut) ────────────────
            fig, ax = plt.subplots(figsize=(10, 7))
            _hist_step_normed(ax, _den_phi[den_sel], BINS_PHI,
                              DEN_COLOR, 'Den (all reco tau)',  '--')
            _hist_step_normed(ax, _num_phi[num_sel], BINS_PHI,
                              NUM_COLOR, 'Num (fake + HLT)',    '-')
            _decorate(ax, rf'Tau $\phi$  ($p_T>{pt_cut}$ GeV)',
                      (BINS_PHI[0], BINS_PHI[-1]))
            plt.tight_layout()
            _save_or_show(fig, save_dir, f"{tag}_numden_phi.png")

In [ ]:
    # ── 6) Num / Den raw histograms ────────────────────────────
    run_all_numden(
        num_pt_reco_150,  den_pt_reco_150,
        num_eta_reco_150, den_eta_reco_150,
        num_phi_reco_150, den_phi_reco_150,
        num_dm_150,       den_dm_150,
        num_njets_150,    den_njets_150,
        pt_cut=150,
        save_dir=SAVE_DIR,
    )